# **7-Day Data Camp - Final project**

## Part 2 - Data Exploration using Python

Import libraries

In [ ]:
from google.colab import files
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

Task 5: Load the dataset into Python

In [ ]:
uploaded = files.upload()

In [ ]:
df = pd.read_csv("file1.csv")
df.head()

Task 6: Perform
- Dataset structire analysis → shape, columns and info to get the basic information about the dataset
  - shape: (rows, columns)
  - columns: name of columns
  - info: type of datum, number of rows, number of missing data

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.info()

- Summary statistics
  - describe, to get basic statistics about the numerical columns;
  - value_counts, to get unique values for categorical columns

In [ ]:
df.describe()

In [ ]:
df[['Country', 'City']].value_counts().sort_index(ascending=True)

In [ ]:
df[['Category', 'Sub_Category']].value_counts().sort_index(ascending=False)

In [ ]:
df[['Segment', 'Sales_Channel', 'Payment_Mode']].value_counts().sort_index(ascending=True)

- Missing value check → isnull, to get the number of missing values

In [ ]:
df.isnull().sum()

- Duplicate detection → duplicated, to get the number of duplicated rows

In [ ]:
df.duplicated().sum()

Looking at the dataset, it is possible to see some inconsistencies in the category and subcategory fields: the pairing is not correct. Therefore we can hardcore every category and subcategory correctly to fix this.

In [ ]:
category_remapping = {
    "Snacks" : "Groceries",
    "Phones" : "Electronics",
    "Cosmetics" : "Beauty",
    "Fitness" : "Sports",
    "Chairs" : "Furniture",
    "Shirts" : "Clothing"
}

# Creating a copy to avoir DettinWithCopyWarning and preserve the original eventually
df_corrected = df.copy()

# Apply the remapping
# Iterate through each row and apply the remapping if the sub_category is in the mapping dictionary
# Only reassing if the current category is NOT the intended category
for index, row in df_corrected.iterrows():
    sub_cat = row["Sub_Category"]
    cat = row["Category"]
    if sub_cat in category_remapping and cat != category_remapping[sub_cat] :
        df_corrected.loc[index, "Category"] = category_remapping[sub_cat]

# Update the dataset
df = df_corrected

In [ ]:
df[['Category', 'Sub_Category']].value_counts().sort_index(ascending=False)

In [ ]:
df.duplicated().sum()

In [ ]:
# Saving the data cleaned at the end of ETL analysis
df.to_csv("file1_corrected_dataset.csv", index = False)
files.download("file1_corrected_dataset.csv")

Task 7 - Analyse
- Distribution of key variavles
- Relationships between variables

In [ ]:
# Istograms to get the behaviour of each quantitative feature

fig, axes = plt.subplots(1, 4, figsize=(20,5))
axes = axes.flatten()

# Sales
axes[0].hist(df['Sales'], bins=20)
axes[0].set_title("Sales Distribution")
axes[0].set_xlabel("Sales")
axes[0].set_ylabel("Frequency")

# Profit
axes[1].hist(df['Profit'], bins=20)
axes[1].set_title("Profit Distribution")
axes[1].set_xlabel("Profit")
axes[1].set_ylabel("Frequency")

# Quantity
axes[2].hist(df['Quantity'], bins=14)
axes[2].set_title("Quantity Distribution")
axes[2].set_xlabel("Quantity")
axes[2].set_ylabel("Frequency")

# Discount
axes[3].hist(df['Discount'], bins=20)
axes[3].set_title("Discount Distribution")
axes[3].set_xlabel("Discount")
axes[3].set_ylabel("Frequency")

plt.tight_layout()
plt.show()

In [ ]:
# Bar plots to get the statistics of each quantitative feature: typical range, central tendency, variability and outliers

fig, axes = plt.subplots(1, 4, figsize=(20,5))
axes = axes.flatten()

# Sales
axes[0].boxplot(df['Sales'])
axes[0].set_title("Sales Boxplot")
axes[0].set_ylabel("Sales")

# Profit
axes[1].boxplot(df['Profit'])
axes[1].set_title("Profit Boxplot")
axes[1].set_ylabel("Profit")

# Quantity
axes[2].boxplot(df['Quantity'])
axes[2].set_title("Quantity Boxplot")
axes[2].set_ylabel("Quantity")

# Discount
axes[3].boxplot(df['Discount'])
axes[3].set_title("Discount Boxplot")
axes[3].set_ylabel("Discount")

plt.tight_layout()
plt.show()

In [ ]:
# Function to find outliers using IQR method
def find_outliers_iqr(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    bound_lower = Q1 - 1.5 * IQR
    bound_upper = Q3 + 1.5 * IQR
    outliers = df[(df[column] < bound_lower) | (df[column] > bound_upper)]
    return outliers

# Applying function to Profit feature, as it is the only one which presents outliers in the boxplot
profit_outliers = find_outliers_iqr(df, "Profit")
if not profit_outliers.empty:
    display(profit_outliers)
else:
    print("No significant profit outliers found based on IQR method")

# Further analyses on these outliers may identify commonities. Use value_count to find them; e.g.:
profit_outliers[['Category', 'Sub_Category']].value_counts().sort_index(ascending=False)

In [ ]:
# Scatterplot to investigate the relationships between quantitative feature

fig, axes = plt.subplots(1, 3, figsize=(15,5))
axes = axes.flatten()

# Sales vs Profit
axes[0].scatter(df['Profit'], df['Sales'])
axes[0].set_title("Sales vs Profit")
axes[0].set_xlabel("Profit")
axes[0].set_ylabel("Sales")

# Sales vs Quantity
axes[1].scatter(df['Quantity'], df['Sales'])
axes[1].set_title("Sales vs Quantity")
axes[1].set_xlabel("Quantity")
axes[1].set_ylabel("Sales")

# Sales vs Discount
axes[2].scatter(df['Discount'], df['Sales'])
axes[2].set_title("Sales vs Discount")
axes[2].set_xlabel("Discount")
axes[2].set_ylabel("Sales")

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15,5))
axes = axes.flatten()

# Profit vs Sales
axes[0].scatter(df['Sales'], df['Profit'])
axes[0].set_title("Profit vs Sales")
axes[0].set_xlabel("Sales")
axes[0].set_ylabel("Profit")

# Profit vs Sales
axes[1].scatter(df['Quantity'], df['Profit'])
axes[1].set_title("Profit vs Quantity")
axes[1].set_xlabel("Quantity")
axes[1].set_ylabel("Profit")

# Profit vs Sales
axes[2].scatter(df['Discount'], df['Profit'])
axes[2].set_title("Profit vs Discount")
axes[2].set_xlabel("Discount")
axes[2].set_ylabel("Profit")

plt.tight_layout()
plt.show()

Quantity and Discount seems not correlated with Profit and Sales as there is no trend in the scatterplots.

Before proceeding with the predictive analysis, I decided to include the Order_Date feature to eventually improve the ML models if trends are present.
To do so, I converted the column into a date format and created some additional columns.

In [ ]:
df['Order_Date'] = pd.to_datetime(df['Order_Date'])

df['Year'] = df['Order_Date'].dt.year
df['Month'] = df['Order_Date'].dt.month
df['Day'] = df['Order_Date'].dt.day

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15,5))
axes = axes.flatten()

# Sales vs Profit
axes[0].scatter(df['Year'], df['Sales'])
axes[0].set_title("Sales vs Year")
axes[0].set_xlabel("Year")
axes[0].set_ylabel("Sales")

# Sales vs Quantity
axes[1].scatter(df['Month'], df['Sales'])
axes[1].set_title("Sales vs Month")
axes[1].set_xlabel("Month")
axes[1].set_ylabel("Sales")

# Sales vs Discount
axes[2].scatter(df['Day'], df['Sales'])
axes[2].set_title("Sales vs Day")
axes[2].set_xlabel("Day")
axes[2].set_ylabel("Sales")

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15,5))
axes = axes.flatten()

# Sales vs Profit
axes[0].scatter(df['Year'], df['Profit'])
axes[0].set_title("Profit vs Year")
axes[0].set_xlabel("Year")
axes[0].set_ylabel("Profit")

# Sales vs Quantity
axes[1].scatter(df['Month'], df['Profit'])
axes[1].set_title("Profit vs Month")
axes[1].set_xlabel("Month")
axes[1].set_ylabel("Profit")

# Sales vs Discount
axes[2].scatter(df['Day'], df['Profit'])
axes[2].set_title("Profit vs Day")
axes[2].set_xlabel("Day")
axes[2].set_ylabel("Profit")

plt.tight_layout()
plt.show()

Order date seems not correlated with Profit and Sales as there is no trend in the scatterplots.

Correlation Matrix Analysis

In [ ]:
df_numerical = df[["Sales", "Quantity", "Discount", "Profit"]]
correlation_matrix = df_numerical.corr()
display(correlation_matrix)

# Heat map of correlation matrix
plt.figure(figsize = (8,6))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Correlation matrix of numerical features")
plt.show()

The most notable correlation is the positive correlation between Profit and Sales, which is weak. Other variables suggest no correlation at all between them.

## Part 3 - Predictive Analysis (Machine Learning)

Task 8: Select features and target value

I decided to predict Profit from Sales, which is a more useful information for a company. As the other quantitative features seems not correlated to Profit, I omitted them to avoid noise in the predictions.

In [ ]:
x = df[['Sales']]
y = df['Profit']

Task 9: Split data into training and test sets

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.2)

Task 10: Build a Linear Regression model

In [ ]:
model = LinearRegression()
model.fit(x_train, y_train)

print(f"Model coefficients: {model.coef_}")
print(f"Model intercept: {model.intercept_}")

Task 11: Generate predictions

In [ ]:
lr_predictions = model.predict(x_test)

In [ ]:
# Calculate evaluation metrics
mae = mean_absolute_error(y_test, lr_predictions)
mse = mean_squared_error(y_test, lr_predictions)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, lr_predictions)

print(f"Model Evaluation on Test Set:")
print(f"Mean Absolute Error: {mae:.2f}")
print(f"Mean Squared Error: {mse:.2f}")
print(f"Root Mean Squared Error: {rmse:.2f}")
print(f"R-squared Score: {r2:.2f}")

Task 13: Compare actual vs predicted values for Linear Regression model

In [ ]:
comparison = pd.DataFrame({"Actual":y_test, "Predicted":lr_predictions})
comparison.head()

In [ ]:
plt.scatter(y_test, lr_predictions)
plt.xlabel("Actual Profit")
plt.ylabel("Predicted Profit")
plt.title("Linear Regression model - Actual vs Predicted")
plt.show()

Task 12: Optional - Apply a Decision Tree model

In [ ]:
tree_model = DecisionTreeRegressor()
tree_model.fit(x_train, y_train)

In [ ]:
tree_predictions = tree_model.predict(x_test)

In [ ]:
comparison = pd.DataFrame({"Actual":y_test, "Predicted":tree_predictions})
comparison.head()

Task 13: Compare Actual vs Predicted values

In [ ]:
plt.scatter(y_test, tree_predictions)
plt.xlabel("Actual Profit")
plt.ylabel("Predicted Profit")
plt.title("Decision Tree model - Actual vs Predicted")
plt.show()

In [ ]:
plt.scatter(y_test, lr_predictions, color="blue", label="Linear Regression")
plt.scatter(y_test, tree_predictions, color="red", label="Decision Tree")
plt.xlabel("Actual Profit")
plt.ylabel("Predicted Profit")
plt.title("Comparison between models")
plt.legend()
plt.show()

Linear Regression model seems to predict better the outcomes than the Decision Tree model. Nevertheless, the model need improvements to get more precise predictions.

## Part 4 - Insights & Business Recommendations

- Which category performs best and why?

The category which performs best is sport.

-	Which region shows growth potential?

India shows growth potential as the profits increased in the last years, while in other countries the profits were stable or decreased.

- What factors influence sales the most?

Limiting the analysis to quantitative features and based on the scatterplots above, only the profit influences the sales, as the other variables do not present trends.

Considering the qualitative features, the sales are mostly influenced by country and category, which present trends (as shown in the graphs in Tableau).

-	How accurate is your prediction model?

Both models (Linear Regression and Decision Tree) are not very accurate, as shown in the last scatterplot. More features are needed to improve them.

To improve the models, more variables are needed and, therefore, efforts should be done to include in some ways the regions of the sales and the categories of products.

-	What business recommendations would you give?

I would suggest to avoid too high discounts as they reduce the profit and limit them only to special occasions. Based on recent trends, India shows strong potential for profit growth, making it a promising market for increased investment. USA has great profits, especially for Sport category, making it a valuable market.